In [1]:
from pathlib import Path
import re
import html
import unicodedata
import urllib.request

import numpy as np
import pandas as pd
from underthesea import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import save_npz
import joblib

In [4]:
# 1. CONFIGURATION

PROJECT_DIR = Path(".")

INPUT_PATH = PROJECT_DIR / "news_dataset.csv"

STOPWORDS_PATH = (
    PROJECT_DIR / "vietnamese-stopwords-dash.txt"
)

OUTPUT_DIR = PROJECT_DIR / "data_preprocessed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = OUTPUT_DIR / "news_dataset_preprocessed.csv"

STOPWORDS_PATH = PROJECT_DIR / "vietnamese-stopwords-dash.txt"

In [5]:
# Keep numbers as a common token by default.
# Set False if you prefer to remove all numbers.
REPLACE_NUMBERS = True

# Combine title and article body. Set False to use Content only.
USE_TITLE = True

# TF-IDF settings; fit only on train data.
MAX_FEATURES = 50000
NGRAM_RANGE = (1, 2)
MIN_DF = 2
MAX_DF = 0.95

In [6]:
# 2. LOAD DATA

# The supplied CSV has an extra first row containing the intended
# column names. header=1 skips that row.
df = pd.read_csv(INPUT_PATH, header=1, encoding="utf-8-sig")

required_cols = {"ID", "Title", "Content", "Label", "Split"}
missing_cols = required_cols - set(df.columns)
if missing_cols:
    raise ValueError(f"Missing required columns: {sorted(missing_cols)}")

df["Title"] = df["Title"].fillna("").astype(str)
df["Content"] = df["Content"].fillna("").astype(str)
df["Label"] = df["Label"].astype(str).str.strip()
df["Split"] = df["Split"].astype(str).str.strip().str.lower()

valid_splits = {"train", "val", "test"}
unknown_splits = set(df["Split"].unique()) - valid_splits
if unknown_splits:
    raise ValueError(f"Unexpected Split values: {unknown_splits}")

if df["ID"].duplicated().any():
    raise ValueError("ID is not unique.")
if df["Label"].isna().any() or df["Content"].isna().any():
    raise ValueError("Label or Content contains missing values.")

print("Dataset shape:", df.shape)
print("\nSplit distribution:")
print(df["Split"].value_counts())
print("\nLabel distribution:")
print(df["Label"].value_counts())

Dataset shape: (5898, 14)

Split distribution:
Split
train    4129
val       886
test      883
Name: count, dtype: int64

Label distribution:
Label
Thế giới     810
Sức khỏe     750
Pháp luật    746
Du lịch      746
Kinh tế      740
Giáo dục     727
Thể thao     715
Công nghệ    664
Name: count, dtype: int64


In [7]:
valid_splits = {"train", "val", "test"}
unknown_splits = set(df["Split"].unique()) - valid_splits
if unknown_splits:
    raise ValueError(f"Unexpected Split values: {unknown_splits}")

if df["ID"].duplicated().any():
    raise ValueError("ID is not unique.")
if df["Label"].isna().any() or df["Content"].isna().any():
    raise ValueError("Label or Content contains missing values.")

print("Dataset shape:", df.shape)
print("\nSplit distribution:")
print(df["Split"].value_counts())
print("\nLabel distribution:")
print(df["Label"].value_counts())

Dataset shape: (5898, 14)

Split distribution:
Split
train    4129
val       886
test      883
Name: count, dtype: int64

Label distribution:
Label
Thế giới     810
Sức khỏe     750
Pháp luật    746
Du lịch      746
Kinh tế      740
Giáo dục     727
Thể thao     715
Công nghệ    664
Name: count, dtype: int64


In [8]:
with open(STOPWORDS_PATH, "r", encoding="utf-8") as f:
    stopwords = {
        line.strip().lower()
        for line in f
        if line.strip() and not line.lstrip().startswith("#")
    }

# Preserve negation terms: removing them can change sentence meaning.
# The exact entries may or may not exist in the downloaded list.
NEGATION_WORDS = {"không", "chưa", "chẳng", "chả", "không_thể"}
stopwords -= NEGATION_WORDS

print("\nNumber of stopwords:", len(stopwords))


Number of stopwords: 1942


In [9]:
# 4. TEXT CLEANING
URL_PATTERN = re.compile(r"(https?://\S+|www\.\S+)", flags=re.IGNORECASE)
EMAIL_PATTERN = re.compile(r"\b[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}\b")
HTML_TAG_PATTERN = re.compile(r"<[^>]+>")
# Remove common emoji / pictographic ranges.
EMOJI_PATTERN = re.compile(
    "["
    "\U0001F300-\U0001F5FF"
    "\U0001F600-\U0001F64F"
    "\U0001F680-\U0001F6FF"
    "\U0001F700-\U0001F77F"
    "\U0001F780-\U0001F7FF"
    "\U0001F800-\U0001F8FF"
    "\U0001F900-\U0001F9FF"
    "\U0001FA00-\U0001FAFF"
    "\U00002700-\U000027BF"
    "\U00002600-\U000026FF"
    "]+",
    flags=re.UNICODE,
)

def clean_text(text):
    """Clean HTML, URLs, emoji, punctuation and normalize whitespace."""
    text = "" if pd.isna(text) else str(text)
    text = html.unescape(text)
    text = unicodedata.normalize("NFC", text)
    text = text.lower()

    text = URL_PATTERN.sub(" ", text)
    text = EMAIL_PATTERN.sub(" ", text)
    text = HTML_TAG_PATTERN.sub(" ", text)
    text = EMOJI_PATTERN.sub(" ", text)

    if REPLACE_NUMBERS:
        text = re.sub(r"\d+(?:[.,]\d+)*", " numtoken ", text)
    else:
        text = re.sub(r"\d+(?:[.,]\d+)*", " ", text)

    # Keep Vietnamese letters, whitespace and underscore.
    # Underscore is useful after Vietnamese word segmentation.
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = re.sub(r"_+", "_", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


In [11]:
import re
import html
import unicodedata

from pathlib import Path

import pandas as pd
from underthesea import word_tokenize


# ==================================================
# 1. PATH CONFIGURATION
# ==================================================


STOPWORDS_PATH = (
    PROJECT_DIR / "vietnamese-stopwords-dash.txt"
)

OUTPUT_DIR = PROJECT_DIR / "data_preprocessed"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

OUTPUT_PATH = (
    OUTPUT_DIR / "news_dataset_preprocessed.csv"
)


# ==================================================
# 2. REGEX PATTERNS
# ==================================================

_HTML_TAG = re.compile(r"<[^>]+>")

_URL = re.compile(
    r"(https?://\S+|www\.\S+)",
    re.IGNORECASE
)

_EMAIL = re.compile(
    r"\S+@\S+\.\S+"
)

_SPACES = re.compile(
    r"\s+"
)

# Token phải chứa ít nhất một chữ cái.
# Cho phép chữ số và dấu "_" trong token.
# Ví dụ: u17, 5g, việt_nam.
_WORD = re.compile(
    r"(?=[^\W_]*[^\W\d_])[^\W_]+(?:_[^\W_]+)*",
    re.UNICODE
)


# ==================================================
# 3. LOAD STOPWORDS
# ==================================================

def load_stopwords(
    path: Path = STOPWORDS_PATH
) -> set[str]:

    if not path.exists():
        raise FileNotFoundError(
            f"Không tìm thấy file stopwords: {path}"
        )

    words = set()

    for line in path.read_text(
        encoding="utf-8"
    ).splitlines():

        word = line.strip().lower()

        if word and not word.startswith("#"):
            words.add(word)

    # Không loại bỏ các từ phủ định quan trọng
    negation_words = {
        "không",
        "chưa",
        "chẳng"
    }

    words -= negation_words

    return words


# ==================================================
# 4. REMOVE MARKUP
# ==================================================

def strip_markup(text) -> str:
    """
    Giai đoạn 1:
    Chỉ loại bỏ thành phần kỹ thuật.

    Giữ lại:
    - Chữ hoa
    - Dấu câu
    - Số

    Mục đích:
    Bảo toàn thông tin cho bước tách từ.
    """

    if not isinstance(text, str):
        return ""

    # Chuẩn hóa Unicode tiếng Việt
    text = unicodedata.normalize(
        "NFC",
        text
    )

    # Giải mã HTML entities
    text = html.unescape(text)

    # Loại bỏ HTML tags
    text = _HTML_TAG.sub(" ", text)

    # Loại bỏ URL
    text = _URL.sub(" ", text)

    # Loại bỏ email
    text = _EMAIL.sub(" ", text)

    # Giữ ranh giới giữa các đoạn
    text = text.replace(
        "\n",
        " . "
    )

    return _SPACES.sub(
        " ",
        text
    ).strip()


# ==================================================
# 5. WORD SEGMENTATION
# ==================================================

def tokenize(text: str) -> list[str]:
    """
    Tách từ tiếng Việt bằng underthesea.

    Kết quả ví dụ:
        Đội tuyển bóng đá Việt Nam

    Thành:
        Đội_tuyển bóng_đá Việt_Nam
    """

    if not text:
        return []

    segmented = word_tokenize(
        text,
        format="text"
    )

    return segmented.split()


# ==================================================
# 6. KEEP VALID WORDS
# ==================================================

def keep_words(tokens: list[str]) -> list[str]:
    """
    Giai đoạn 2:
    - Loại dấu câu
    - Loại ký hiệu đặc biệt
    - Loại token chỉ chứa số
    - Chuyển chữ thường

    Giữ các token có chữ và số như U17, 5G.
    """

    cleaned_tokens = []

    for token in tokens:

        token = token.strip()

        # Chỉ giữ token hợp lệ
        if _WORD.fullmatch(token):

            cleaned_tokens.append(
                token.lower()
            )

    return cleaned_tokens


# ==================================================
# 7. REMOVE STOPWORDS
# ==================================================

def remove_stopwords(
    tokens: list[str],
    stopwords: set[str],
    min_len: int = 2
) -> list[str]:
    """
    Loại bỏ stopwords và token quá ngắn.

    min_len không tính dấu "_" trong từ ghép.
    """

    return [
        token
        for token in tokens

        if token not in stopwords

        and len(
            token.replace("_", "")
        ) >= min_len
    ]


# ==================================================
# 8. PREPROCESS TEXT
# ==================================================

def preprocess_text(
    text: str,
    stopwords: set[str]
) -> list[str]:
    """
    Văn bản thô -> danh sách token sạch.
    """

    # Bước 1: Loại bỏ markup
    text = strip_markup(text)

    # Bước 2: Tách từ
    tokens = tokenize(text)

    # Bước 3: Lọc dấu câu, số độc lập,
    # chuẩn hóa chữ thường
    tokens = keep_words(tokens)

    # Bước 4: Loại stopwords
    tokens = remove_stopwords(
        tokens,
        stopwords
    )

    return tokens


# ==================================================
# 9. BUILD CLEAN DATASET
# ==================================================

def build_clean() -> pd.DataFrame:

    # Đọc dữ liệu
    # header=1 nếu file có một dòng tên cột giả ở đầu.
    # Nếu dòng đầu tiên đã là header thật, đổi thành 0.
    df = pd.read_csv(
        INPUT_PATH,
        encoding="utf-8",
        header=1
    )

    required_columns = [
        "ID",
        "Title",
        "Content",
        "Label",
        "Split"
    ]

    missing = [
        col for col in required_columns
        if col not in df.columns
    ]

    if missing:
        raise ValueError(
            f"Thiếu các cột bắt buộc: {missing}"
        )

    # Load stopwords
    stopwords = load_stopwords()

    print(f"Đọc {len(df)} bài viết")
    print(f"Số stopwords: {len(stopwords)}")
    print("Tokenizer: underthesea")

    # Kết hợp tiêu đề và nội dung
    raw_text = (
        df["Title"].fillna("").astype(str)
        + ". "
        + df["Content"].fillna("").astype(str)
    )

    # Tiền xử lý
    df["Processed_Text"] = raw_text.apply(
        lambda text: " ".join(
            preprocess_text(
                text,
                stopwords
            )
        )
    )

    # Kiểm tra văn bản rỗng
    empty = (
        df["Processed_Text"]
        .str.strip()
        .eq("")
        .sum()
    )

    if empty:
        print(
            f"CẢNH BÁO: {empty} bài rỗng "
            "sau tiền xử lý"
        )

    # Thống kê token
    token_counts = (
        df["Processed_Text"]
        .str.split()
        .str.len()
    )

    print("\nPREPROCESSING SUMMARY")
    print("---------------------")

    print("Số bài:", len(df))
    print("Số bài rỗng:", empty)

    print(
        "Token/bài trung bình:",
        round(token_counts.mean(), 2)
    )

    print(
        "Token/bài nhỏ nhất:",
        token_counts.min()
    )

    print(
        "Token/bài lớn nhất:",
        token_counts.max()
    )

    # Lưu dataset
    df.to_csv(
        OUTPUT_PATH,
        index=False,
        encoding="utf-8-sig"
    )

    print("\nĐã lưu dữ liệu:")
    print(OUTPUT_PATH)

    return df


# ==================================================
# 10. MAIN
# ==================================================

if __name__ == "__main__":

    final_df = build_clean()

    print("\nVí dụ kết quả:")
    print(
        final_df[
            [
                "Title",
                "Processed_Text"
            ]
        ].head(3).to_string(index=False)
    )


Đọc 5898 bài viết
Số stopwords: 1943
Tokenizer: underthesea

PREPROCESSING SUMMARY
---------------------
Số bài: 5898
Số bài rỗng: 0
Token/bài trung bình: 275.21
Token/bài nhỏ nhất: 40
Token/bài lớn nhất: 2638

Đã lưu dữ liệu:
data_preprocessed\news_dataset_preprocessed.csv

Ví dụ kết quả:
                                                      Title                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          

In [19]:
# Bước 1: Chạy hàm tiền xử lý
df = build_clean()

# Bước 2: Kiểm tra kết quả
print(df[["Title", "Processed_Text"]].head(3))

# Bước 3: Chọn các cột cần xuất
output_df = df[
    ["ID", "Label", "Split", "Processed_Text"]
].copy()

# Bước 4: Xuất CSV
output_df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Đã lưu:", OUTPUT_PATH)

Đọc 5898 bài viết
Số stopwords: 1943
Tokenizer: underthesea

PREPROCESSING SUMMARY
---------------------
Số bài: 5898
Số bài rỗng: 0
Token/bài trung bình: 275.21
Token/bài nhỏ nhất: 40
Token/bài lớn nhất: 2638

Đã lưu dữ liệu:
data_preprocessed\news_dataset_preprocessed.csv
                                               Title  \
0         Mỗi lượng vàng miếng giảm tới 4 triệu đồng   
1  COP31 có thể bỏ mục tiêu cam kết hạn chế than,...   
2  APEC mở cơ hội để Việt Nam đón vốn chất lượng cao   

                                      Processed_Text  
0  vàng miếng triệu đồng sjc hạ giá vàng miếng tr...  
1  cop31 mục_tiêu cam_kết hạn_chế than dầu khí_đố...  
2  apec việt_nam đón vốn chất_lượng apec thâm_quy...  
Đã lưu: data_preprocessed\news_dataset_preprocessed.csv
